# ARS Notebook 03b — MATH-500 OOD Evaluation

Evaluate the four paired arms on the selected number of MATH-500 test questions (all 500 by default). The notebook runs 0-shot first and packages it, then runs 2-shot and packages it separately. Each condition uses the same seeded question IDs and the four arms share a prompt schedule. MATH-Verify scores symbolic answers by mathematical equivalence, with a literal-answer fallback for text-valued answers.


## Required inputs

Attach one complete Stage B `ars_training_bundle.zip` or one extracted Stage B input tree with its root `manifest.json`. It must contain the fixed adapter, route bank, router, both norm calibrations, and the selected static-bank action. No Stage A bundle, older checkpoint, or rollout files are needed.

The notebook installs pinned `math-verify==0.9.0` plus its supported ANTLR 4.13.2 runtime if missing. Kaggle internet must be enabled for package installation and dataset access. For a resumed session, attach either or both result ZIPs made by this notebook; it verifies provenance, reuses completed conditions, and emits separate 0-shot and 2-shot ZIPs.


In [ ]:
import hashlib, json, os, shutil, subprocess, sys
from pathlib import Path

os.environ["TOKENIZERS_PARALLELISM"] = "false"
MATH_VERIFY_VERSION = "0.9.0"
MATH_VERIFY_ANTLR_VERSION = "4.13.2"
try:
    from importlib.metadata import version as package_version
    installed_math_verify = package_version("math-verify")
except Exception:
    installed_math_verify = None
try:
    installed_antlr = package_version("antlr4-python3-runtime")
except Exception:
    installed_antlr = None
if installed_math_verify != MATH_VERIFY_VERSION or installed_antlr != MATH_VERIFY_ANTLR_VERSION:
    subprocess.run([
        sys.executable, "-m", "pip", "install",
        f"math-verify[antlr4_13_2]=={MATH_VERIFY_VERSION}",
    ], check=True)
from math_verify import parse as parse_math_answer, verify as verify_math_answer
from importlib.metadata import version as package_version
assert package_version("math-verify") == MATH_VERIFY_VERSION
assert package_version("antlr4-python3-runtime") == MATH_VERIFY_ANTLR_VERSION
if not verify_math_answer(parse_math_answer(r"\frac{1}{2}"), parse_math_answer("0.5")):
    raise RuntimeError("Math-Verify symbolic-equivalence self-check failed.")
REPO_URL = "https://github.com/Abdullah182155/adaptive-residual-steering.git"
REPO_DIR = Path("/kaggle/working/adaptive-residual-steering")
WORK_ROOT = Path("/kaggle/working")
INPUT_ROOT = Path("/kaggle/input")
if (REPO_DIR / ".git").exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", "development"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "development"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", "development"], check=True)
else:
    subprocess.run(["git", "clone", "--branch", "development", REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))
from scripts.research_notebook_utils import (
    extract_zip_once, locate_named_file, merge_result_bundle, pack_files,
    resolve_artifacts, run_live, sha256_file, atomic_write_json,
)

# Keep the training and evaluation revisions explicit. A stale script snapshot
# inside the training bundle must not replace the current benchmark runner.
EVALUATION_CODE_COMMIT = subprocess.run(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"],
    check=True, capture_output=True, text=True,
).stdout.strip()
training_zips = sorted(INPUT_ROOT.rglob("ars_training_bundle.zip"))
if len(training_zips) > 1:
    raise ValueError(f"Attach one training bundle only; found: {[str(p) for p in training_zips]}")
TRAINING_ZIP = training_zips[0] if training_zips else None
TRAINING_INPUT = None
TRAINING_MANIFEST = {}
if TRAINING_ZIP:
    ARTIFACT_ROOT = extract_zip_once(TRAINING_ZIP, WORK_ROOT / "ars_training_artifacts")
    TRAINING_INPUT = TRAINING_ZIP
    manifest_path = ARTIFACT_ROOT / "manifest.json"
    if manifest_path.is_file():
        TRAINING_MANIFEST = json.loads(manifest_path.read_text(encoding="utf-8"))
else:
    manifests = []
    for candidate in INPUT_ROOT.rglob("manifest.json"):
        try:
            data = json.loads(candidate.read_text(encoding="utf-8"))
        except (OSError, ValueError, TypeError):
            continue
        if data.get("format") == "ars_training_bundle_v1":
            manifests.append((candidate, data))
    if len(manifests) > 1:
        raise ValueError(f"Attach one extracted ARS training bundle only; found: {[str(p) for p, _ in manifests]}")
    if manifests:
        manifest_path, TRAINING_MANIFEST = manifests[0]
        ARTIFACT_ROOT = manifest_path.parent
        TRAINING_INPUT = ARTIFACT_ROOT
    else:
        ARTIFACT_ROOT = INPUT_ROOT
        manifest_path = None

REQUIRED_ARTIFACTS = [
    "fixed_route_adapter.pt", "session3_steering_bank.pt", "session3_router.pt",
    "fixed_route_p99_norms.json", "candidate_norms_p99.json",
    "session3_static_bank_action.json",
]
try:
    ARTIFACTS = resolve_artifacts(ARTIFACT_ROOT, REQUIRED_ARTIFACTS)
    INPUTS_READY = bool(TRAINING_INPUT)
except Exception as exc:
    ARTIFACTS, INPUTS_READY = {}, False
    print("Training bundle is missing or invalid:", exc, flush=True)
if not TRAINING_INPUT:
    INPUTS_READY = False
    print("Attach ars_training_bundle.zip or one extracted input containing its root manifest.json.", flush=True)
elif (
    not TRAINING_MANIFEST.get("training_ready", False)
    or TRAINING_MANIFEST.get("architecture_id") != "ars.fixed_route_low_rank_residual"
    or TRAINING_MANIFEST.get("architecture_version") != "1.0.0"
):
    INPUTS_READY = False
    print("Training bundle is not ready or has an unsupported fixed-adapter architecture.", TRAINING_MANIFEST.get("required_phase_status"), flush=True)
TRAINING_CODE_COMMIT = TRAINING_MANIFEST.get("code_commit")
TRAINING_INPUT_SHA256 = (
    sha256_file(TRAINING_ZIP) if TRAINING_ZIP else
    sha256_file(manifest_path) if TRAINING_INPUT and manifest_path and manifest_path.is_file() else None
)
EXPECTED_ARTIFACT_SHA256 = (
    {
        "fixed_adapter_checkpoint": sha256_file(ARTIFACTS["fixed_route_adapter.pt"]),
        "fixed_adapter_norms": sha256_file(ARTIFACTS["fixed_route_p99_norms.json"]),
        "bank_checkpoint": sha256_file(ARTIFACTS["session3_steering_bank.pt"]),
        "router_checkpoint": sha256_file(ARTIFACTS["session3_router.pt"]),
        "candidate_norms": sha256_file(ARTIFACTS["candidate_norms_p99.json"]),
        "static_bank_action": sha256_file(ARTIFACTS["session3_static_bank_action.json"]),
    }
    if INPUTS_READY else {}
)
ARTIFACT_FINGERPRINT = hashlib.sha256(
    json.dumps(EXPECTED_ARTIFACT_SHA256, sort_keys=True).encode("utf-8")
).hexdigest()[:12] if INPUTS_READY else "missing"
print({
    "repository": str(REPO_DIR),
    "evaluation_code_commit": EVALUATION_CODE_COMMIT,
    "training_code_commit": TRAINING_CODE_COMMIT,
    "training_input": str(TRAINING_INPUT) if TRAINING_INPUT else None,
    "input_mode": "zip" if TRAINING_ZIP else ("extracted_directory" if TRAINING_INPUT else "missing"),
    "artifacts_ready": INPUTS_READY,
    "artifact_fingerprint": ARTIFACT_FINGERPRINT,
    "artifacts": {key: str(value) for key, value in ARTIFACTS.items()},
    "gpus": subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.splitlines(),
}, flush=True)


## Evaluation contract

The public [`HuggingFaceH4/MATH-500`](https://huggingface.co/datasets/HuggingFaceH4/MATH-500) test split contains 500 rows. Set `REQUESTED_QUESTIONS` in the configuration cell to a positive count (for example, 10 for a smoke run), or leave it as `None` to evaluate every available row. Unlike GSM-Plus, MATH-500 contains multi-step competition problems and answers that may be symbolic (fractions, radicals, equations, sets, vectors) or text-valued. The pinned [Hugging Face Math-Verify](https://github.com/huggingface/Math-Verify) library handles symbolic equivalence; text-only gold answers use normalized literal matching from the model’s final answer. A preflight checks all gold answers before inference begins.

The same selected test IDs are reused for 0-shot and 2-shot. To keep the existing ARS evaluation protocol consistent, the 2-shot prompt demonstrations are drawn from the GSM8K training split, not from MATH-500 test rows. This is cross-dataset few-shot context and should be reported as such.


In [ ]:
SEED = 42
REQUESTED_QUESTIONS = None  # None = all available rows; set 10 for a quick end-to-end smoke run.
SHOTS = [0, 2]
DATASET_THIS_SESSION = "math500"
MAX_NEW_TOKENS = 512

from src.evaluation.ood import SPECS as OOD_SPECS, load_ood_rows
from src.evaluation.math_answers import validate_math_gold_answers

if DATASET_THIS_SESSION != "math500":
    raise ValueError("This notebook is intentionally fixed to MATH-500.")
if SHOTS != [0, 2]:
    raise ValueError("The configured study conditions are 0-shot and 2-shot.")
if REQUESTED_QUESTIONS is not None and (
    type(REQUESTED_QUESTIONS) is not int or REQUESTED_QUESTIONS < 1
):
    raise ValueError("REQUESTED_QUESTIONS must be None or a positive integer.")
_current_ood_rows = load_ood_rows(DATASET_THIS_SESSION)
AVAILABLE_QUESTIONS = len(_current_ood_rows)
if AVAILABLE_QUESTIONS < 1:
    raise ValueError("The MATH-500 test split contains no usable rows.")
N_PER_DATASET = (
    AVAILABLE_QUESTIONS
    if REQUESTED_QUESTIONS is None
    else min(REQUESTED_QUESTIONS, AVAILABLE_QUESTIONS)
)
if REQUESTED_QUESTIONS is not None and REQUESTED_QUESTIONS > AVAILABLE_QUESTIONS:
    print(f"Requested {REQUESTED_QUESTIONS} questions; using all {AVAILABLE_QUESTIONS} available rows.", flush=True)
invalid_gold = validate_math_gold_answers(_current_ood_rows)
if invalid_gold:
    raise ValueError(f"Math-Verify could not parse MATH-500 gold answers: {invalid_gold[:10]}")
CURRENT_DATASET_METADATA = {
    "dataset_id": OOD_SPECS[DATASET_THIS_SESSION][0],
    "split": _current_ood_rows[0].get("source_split"),
    "fingerprint": _current_ood_rows[0].get("dataset_fingerprint"),
    "available_rows": len(_current_ood_rows),
    "sampling_seed": SEED,
}
if not INPUTS_READY:
    print("Evaluation is blocked until a complete Stage B training bundle is attached.", flush=True)

RESULTS = WORK_ROOT / "ars_math500_results"
RESULTS.mkdir(parents=True, exist_ok=True)
previous_bundles = sorted({
    path
    for path in INPUT_ROOT.rglob(f"ars_math500_n{N_PER_DATASET}_*shot_seed{SEED}_results.zip")
    if path.is_file()
})
if len(previous_bundles) > len(SHOTS):
    raise ValueError(
        "Attach at most one MATH-500 result ZIP per shot condition; "
        f"found: {[str(path) for path in previous_bundles]}"
    )
for previous_bundle in previous_bundles:
    merge_result_bundle(previous_bundle, RESULTS)

print({
    "dataset": "MATH-500",
    "dataset_id": OOD_SPECS[DATASET_THIS_SESSION][0],
    "source_split": CURRENT_DATASET_METADATA["split"],
    "available_questions": AVAILABLE_QUESTIONS,
    "requested_questions": REQUESTED_QUESTIONS,
    "questions_per_condition": N_PER_DATASET,
    "shots_in_order": SHOTS,
    "seed": SEED,
    "max_new_tokens": MAX_NEW_TOKENS,
    "answer_scoring": "Math-Verify symbolic equivalence plus literal matching for text answers",
    "math_verify_version": MATH_VERIFY_VERSION,
    "antlr_runtime_version": MATH_VERIFY_ANTLR_VERSION,
    "arms": ["baseline", "steer_only", "static_bank", "router"],
    "training_bundle_ready": INPUTS_READY,
    "evaluation_code_commit": EVALUATION_CODE_COMMIT,
    "artifact_fingerprint": ARTIFACT_FINGERPRINT,
    "previous_result_bundles": [str(path) for path in previous_bundles],
}, flush=True)


## Results and interpretation

The notebook displays a four-arm summary after each condition and writes separate result archives:
- `ars_math500_n{N_PER_DATASET}_0shot_seed42_results.zip`
- `ars_math500_n{N_PER_DATASET}_2shot_seed42_results.zip`

The full run uses all available rows by default. Set `REQUESTED_QUESTIONS = 10` for a small smoke run; the ZIP names include the actual question count.

Each ZIP contains the full record, per-question generations, benchmark log, exact-equivalence accuracy and paired comparisons, timing/performance and route-use summaries, run manifest, and code snapshots. Math-Verify uses its symbolic parser for expression-valued answers; literal-only answers use normalized final-answer matching. Report the grader/version and the GSM8K source of 2-shot demonstrations with the results.


In [ ]:
from IPython.display import display
import pandas as pd
from scripts.research_notebook_analysis import build_benchmark_summary

CODE_FILES = [
    "scripts/run_session3_router.py", "scripts/research_notebook_analysis.py",
    "scripts/research_notebook_utils.py", "src/evaluation/benchmark.py",
    "src/evaluation/answers.py", "src/evaluation/ood.py",
    "src/evaluation/generation.py", "src/evaluation/chain.py", "src/evaluation/math_answers.py", "src/configs/config.py",
]

def condition_dir(shot):
    return RESULTS / ARTIFACT_FINGERPRINT / EVALUATION_CODE_COMMIT[:12] / DATASET_THIS_SESSION / f"{shot}shot_n{N_PER_DATASET}"

def record_path(shot):
    return condition_dir(shot) / f"session3_{DATASET_THIS_SESSION}_{N_PER_DATASET}_seed{SEED}.json"

def is_complete(shot):
    path = record_path(shot)
    if not INPUTS_READY or not path.is_file():
        return False
    try:
        record = json.loads(path.read_text(encoding="utf-8"))
        return (
            record.get("suite") == DATASET_THIS_SESSION
            and record.get("test_seed") == SEED
            and len(record.get("test_source_indices", [])) == N_PER_DATASET
            and set(record.get("shots", {})) == {str(shot)}
            and record.get("four_arm_complete") is True
            and record.get("artifact_sha256") == EXPECTED_ARTIFACT_SHA256
            and record.get("dataset_metadata") == CURRENT_DATASET_METADATA
            and record.get("provenance", {}).get("evaluation_code_commit") == EVALUATION_CODE_COMMIT
            and record.get("generation_config", {}).get("max_new_tokens") == MAX_NEW_TOKENS
            and record.get("answer_scoring_method", "").startswith("Math-Verify")
            and record.get("evaluation_metadata", {}).get("math_verify_version") == MATH_VERIFY_VERSION
            and record.get("evaluation_metadata", {}).get("antlr_runtime_version") == MATH_VERIFY_ANTLR_VERSION
        )
    except (OSError, ValueError, TypeError):
        return False

def package_condition(shot):
    out = condition_dir(shot)
    path = record_path(shot)
    if not is_complete(shot):
        raise RuntimeError(f"MATH-500 {shot}-shot did not complete all four arms on {N_PER_DATASET} paired questions.")
    summary = build_benchmark_summary(
        [out], out / "analysis", artifact_sha256=EXPECTED_ARTIFACT_SHA256,
        artifact_fingerprint=ARTIFACT_FINGERPRINT,
        evaluation_code_commit=EVALUATION_CODE_COMMIT,
    )
    metrics = pd.DataFrame(summary["arm_metrics"])
    if summary["record_count"] != 1 or set(metrics["arm"]) != {"baseline", "steer_only", "static_bank", "router"} or set(metrics["n"]) != {N_PER_DATASET}:
        raise RuntimeError(f"Invalid four-arm coverage for MATH-500 {shot}-shot: {metrics[['arm', 'n']].to_dict('records')}")
    print(f"MATH-500 {shot}-shot summary:")
    columns = [
        "shot", "arm", "n", "accuracy_pct", "prediction_parse_rate_pct", "grading_error_count",
        "paired_delta_pp_vs_baseline", "paired_ci95_pp_vs_baseline", "mcnemar_p_value_vs_baseline",
        "mean_generation_latency_seconds", "questions_per_second", "output_tokens_per_second",
        "mean_new_tokens", "router_unique_routes", "router_most_used_route_share", "router_k_counts",
    ]
    display(metrics[[name for name in columns if name in metrics]].round(3))
    paired = pd.DataFrame(summary["paired_comparisons"])
    if not paired.empty:
        display(paired[[name for name in ["reference_arm", "arm", "n", "wins", "losses", "ties", "delta_pp", "ci95_pp", "mcnemar_p_value"] if name in paired]].round(4))
    record = json.loads(path.read_text(encoding="utf-8"))
    atomic_write_json(out / "result_bundle_manifest.json", {
        "format": "ars_single_condition_result_bundle_v1", "dataset": "MATH-500",
        "dataset_id": OOD_SPECS[DATASET_THIS_SESSION][0], "source_split": CURRENT_DATASET_METADATA["split"],
        "dataset_fingerprint": CURRENT_DATASET_METADATA["fingerprint"],
        "requested_questions": N_PER_DATASET, "paired_questions": len(record["test_source_indices"]),
        "shot_count": shot, "seed": SEED, "arms": ["baseline", "steer_only", "static_bank", "router"],
        "four_arm_complete": True, "artifact_fingerprint": ARTIFACT_FINGERPRINT,
        "answer_scoring_method": record.get("answer_scoring_method"),
        "math_verify_version": MATH_VERIFY_VERSION, "antlr_runtime_version": MATH_VERIFY_ANTLR_VERSION,
        "max_new_tokens": MAX_NEW_TOKENS,
        "artifact_sha256": EXPECTED_ARTIFACT_SHA256, "training_code_commit": TRAINING_CODE_COMMIT,
        "evaluation_code_commit": EVALUATION_CODE_COMMIT,
        "summary_files": ["analysis/benchmark_metrics_summary.json", "analysis/benchmark_metrics_summary.csv",
                          "analysis/benchmark_paired_comparisons.csv", "analysis/benchmark_route_usage.csv"],
    })
    files = {
        item: (out.relative_to(RESULTS) / item.relative_to(out)).as_posix()
        for item in out.rglob("*") if item.is_file()
    }
    for relative in CODE_FILES:
        source = REPO_DIR / relative
        if source.is_file():
            files[source] = f"code/{relative}"
    bundle = WORK_ROOT / f"ars_math500_n{N_PER_DATASET}_{shot}shot_seed{SEED}_results.zip"
    pack_files(files, bundle)
    print({"shot": shot, "complete": True, "questions": len(record["test_source_indices"]), "result_zip": str(bundle)}, flush=True)
    return bundle

def run_condition(shot):
    out = condition_dir(shot)
    if is_complete(shot):
        print(f"Reusing complete MATH-500 {shot}-shot condition.", flush=True)
        return package_condition(shot)
    if not INPUTS_READY:
        raise RuntimeError("Attach the complete Stage B training bundle before evaluation.")
    out.mkdir(parents=True, exist_ok=True)
    shutil.copy2(ARTIFACTS["session3_static_bank_action.json"], out / "session3_static_bank_action.json")
    command = [
        sys.executable, "-u", "scripts/run_session3_router.py",
        "--stage", "benchmark", "--skip_gsm8k", "--ood_datasets", DATASET_THIS_SESSION,
        "--ood_n", str(N_PER_DATASET), "--output_dir", str(out),
        "--candidate_layers", "8", "11", "12", "13", "15", "17", "18", "19", "20", "21", "22", "23", "24", "26",
        "--norm_calibration", str(ARTIFACTS["candidate_norms_p99.json"]),
        "--fixed_adapter_norm_calibration", str(ARTIFACTS["fixed_route_p99_norms.json"]),
        "--fixed_adapter_checkpoint", str(ARTIFACTS["fixed_route_adapter.pt"]),
        "--bank_checkpoint", str(ARTIFACTS["session3_steering_bank.pt"]),
        "--router_checkpoint", str(ARTIFACTS["session3_router.pt"]),
        "--bench_seed", str(SEED), "--shots", str(shot), "--generation_max_new_tokens", str(MAX_NEW_TOKENS),
        "--require_all_arms", "--diversify_prompts",
    ]
    code = run_live(command, cwd=REPO_DIR, log_path=out / "benchmark.log")
    path = record_path(shot)
    if code != 0 or not path.is_file():
        raise RuntimeError(f"MATH-500 {shot}-shot run failed (exit={code}); inspect {out / 'benchmark.log'}.")
    record = json.loads(path.read_text(encoding="utf-8"))
    record["evaluation_metadata"] = {"math_verify_version": MATH_VERIFY_VERSION,
        "antlr_runtime_version": MATH_VERIFY_ANTLR_VERSION, "max_new_tokens": MAX_NEW_TOKENS}
    record["provenance"] = {
        "training_code_commit": TRAINING_CODE_COMMIT, "evaluation_code_commit": EVALUATION_CODE_COMMIT,
        "training_input_sha256": TRAINING_INPUT_SHA256, "artifact_fingerprint": ARTIFACT_FINGERPRINT,
        "primary_shots": [0, 2],
        "math_verify_version": MATH_VERIFY_VERSION, "antlr_runtime_version": MATH_VERIFY_ANTLR_VERSION,
        "max_new_tokens": MAX_NEW_TOKENS,
    }
    record["artifact_sha256"] = EXPECTED_ARTIFACT_SHA256
    atomic_write_json(path, record)
    return package_condition(shot)

condition_bundles = {}
for shot in SHOTS:
    condition_bundles[shot] = run_condition(shot)

final_summary = build_benchmark_summary(
    [RESULTS], RESULTS / "analysis", artifact_sha256=EXPECTED_ARTIFACT_SHA256,
    artifact_fingerprint=ARTIFACT_FINGERPRINT, evaluation_code_commit=EVALUATION_CODE_COMMIT,
)
final_metrics = pd.DataFrame(final_summary["arm_metrics"])
print("Final MATH-500 summary across completed conditions:")
if not final_metrics.empty:
    display(final_metrics[[name for name in [
        "shot", "arm", "n", "accuracy_pct", "paired_delta_pp_vs_baseline",
        "paired_ci95_pp_vs_baseline", "mcnemar_p_value_vs_baseline",
        "mean_generation_latency_seconds", "questions_per_second", "output_tokens_per_second",
        "router_unique_routes", "router_most_used_route_share", "router_k_counts",
    ] if name in final_metrics]].round(3))
print({
    "completed_shots": sorted({int(value) for value in final_metrics.get("shot", [])}) if not final_metrics.empty else [],
    "summary_json": str(RESULTS / "analysis" / "benchmark_metrics_summary.json"),
    "summary_csv": str(RESULTS / "analysis" / "benchmark_metrics_summary.csv"),
    "paired_comparisons_csv": str(RESULTS / "analysis" / "benchmark_paired_comparisons.csv"),
    "route_usage_csv": str(RESULTS / "analysis" / "benchmark_route_usage.csv"),
    "separate_result_zips": [str(path) for path in condition_bundles.values()],
    "measurement_definitions": final_summary["measure_definitions"],
}, flush=True)
